# Proyek SkinSense AI : Smart Facial Skin Detection & Personalized Skincare Recommendation


## Menentukan Pertanyaan Bisnis

**- Pertanyaan 1**

Bagaimana distribusi jenis kulit wajah pada dataset citra kulit yang digunakan dalam pengembangan sistem klasifikasi kulit wajah selama proses analisis data proyek SkinSense AI?

**- Pertanyaan 2**

Karakteristik visual apa yang paling membedakan setiap jenis dan kondisi kulit wajah pada dataset citra dalam pengembangan model klasifikasi berbasis Deep Learning?


## Import Semua Packages/Library

In [ ]:
import zipfile
import os
import random
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import numpy as np
from collections import Counter
from PIL import Image
import shutil
import pandas as pd
import seaborn as sns
import scipy.stats as stats
from tqdm import tqdm


## Data Wrangling

### Gathering Data

Dataset citra kulit wajah yang digunakan pada proyek ini diperoleh dari platform Kaggle dan disimpan pada Google Drive agar dapat diakses secara terintegrasi.

Dataset Source: https://www.kaggle.com/datasets/igecko/dataset-klasifikasi-kulit-wajah

Dataset citra wajah digunakan untuk membangun sistem klasifikasi jenis kulit wajah yang mampu mengenali empat kategori utama:

- Acne
- Dry Skin
- Normal Skin
- Oily Skin

Dataset diperoleh dari Kaggle dan dipilih karena menyediakan variasi kondisi kulit yang cukup representatif untuk kebutuhan pelatihan model computer vision.

Mengapa Tahap Ini Penting?

Pada proyek machine learning, kualitas dataset merupakan faktor utama yang mempengaruhi performa model. Menurut prinsip Garbage In, Garbage Out (GIGO), model yang dilatih menggunakan data tidak representatif akan menghasilkan prediksi yang buruk meskipun menggunakan algoritma yang kompleks.

### Evidence

Seluruh dataset diperiksa untuk memastikan:

- Struktur folder sesuai kelas
- Jumlah gambar tiap kelas
- Format file valid
- Tidak terdapat file rusak

### Referensi

Dalam riset Mohammed et al. (2022) menyatakan bahwa kualitas data merupakan faktor fundamental dalam pengembangan model machine learning, karena data yang tidak lengkap, tidak konsisten, atau mengandung noise dapat menurunkan performa dan keandalan model secara signifikan.

In [ ]:
# Path file zip
zip_path = "../data/raw/Dataset_Jenis_Kulit.zip"

# Folder hasil extract
extract_path = "../data/extracted/Dataset_Jenis_Kulit"

# Membuat folder jika belum ada
os.makedirs(extract_path, exist_ok=True)

# Extract dataset
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset berhasil di-extract")

In [ ]:
# Path dataset
dataset_path = "../data/extracted"

# Ambil nama folder / kelas
classes = os.listdir(dataset_path)

# Tampilkan sample image
plt.figure(figsize=(15,10))

for i, kelas in enumerate(classes[:4]):
    class_path = os.path.join(dataset_path, kelas)

    # Ambil random image
    image_name = random.choice(os.listdir(class_path))
    image_path = os.path.join(class_path, image_name)

    # Buka gambar
    img = Image.open(image_path)

    # Plot
    plt.subplot(2,2,i+1)
    plt.imshow(img)
    plt.title(kelas)
    plt.axis('off')

plt.tight_layout()
plt.show()

### Assessing Data

#### Tujuan

Melakukan audit kualitas data sebelum digunakan untuk proses pelatihan model.

Parameter yang diperiksa:

- Struktur Dataset: Memastikan seluruh kelas dapat dikenali sistem.

- Ukuran dan Resolusi Gambar: Memastikan gambar memiliki kualitas yang cukup untuk proses ekstraksi fitur.

- Pixel Distribution: Memastikan nilai piksel berada pada rentang yang valid.

- Corrupted Image Detection: Mengidentifikasi file yang tidak dapat dibuka atau mengalami kerusakan.

- Duplicate Detection: Mengidentifikasi gambar yang identik atau hampir identik menggunakan perceptual hashing.

#### Mengapa Dilakukan?

Data duplikat dapat menyebabkan:

- Data leakage
- Overfitting
- Evaluasi model menjadi bias

Studi oleh Rosenblatt, M., et al. (2024) menunjukkan bahwa data leakage dapat menyebabkan performa prediksi model tampak lebih baik dari kondisi sebenarnya, sehingga evaluasi menjadi tidak reliabel untuk data dunia nyata.

In [ ]:
#cek kelas
os.listdir("../data/extracted")

In [ ]:
#cek jumlah per kelas
dataset_path = "../data/extracted"

for folder in os.listdir("../data/extracted"):
    path = os.path.join(dataset_path, folder)
    if os.path.isdir(path):
        print(folder, ":", len(os.listdir(path)))

In [ ]:
#cek jumlah total dataset
dataset_path = "../data/extracted"
total_images = 0

for folder in os.listdir(dataset_path):
    path = os.path.join(dataset_path, folder)
    if os.path.isdir(path):
        total_images += len(os.listdir(path))

print("Total dataset:", total_images)

In [ ]:
#cek ukuran gambar
sizes = []

for folder in os.listdir("../data/extracted"):
    folder_path = os.path.join(dataset_path, folder)

    if os.path.isdir(folder_path):
        for img_name in os.listdir(folder_path):

            img_path = os.path.join(folder_path, img_name)
            img = cv2.imread(img_path)

            if img is not None:
                sizes.append(img.shape)

print("Contoh ukuran gambar:", sizes[:5])

In [ ]:
#cek resolusi
dataset_path = "../data/extracted"

heights = []
widths = []

for folder in os.listdir("../data/extracted"):
    folder_path = os.path.join("../data/extracted", folder)

    if os.path.isdir(folder_path):
        for img_name in os.listdir(folder_path):
            img_path = os.path.join(folder_path, img_name)

            img = cv2.imread(img_path)

            if img is not None:
                h, w, _ = img.shape
                heights.append(h)
                widths.append(w)


In [ ]:
print("Resolusi minimum:")
print(min(widths), "x", min(heights))

print("\nResolusi maksimum:")
print(max(widths), "x", max(heights))

print("\nResolusi rata-rata:")
print(int(np.mean(widths)), "x", int(np.mean(heights)))

In [ ]:
#cek jumlah file berdasarkan format
dataset_path = "../data/extracted"

formats = []

for folder in os.listdir("../data/extracted"):
    folder_path = os.path.join("../data/extracted", folder)

    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            ext = os.path.splitext(file)[1].lower()  # ambil ekstensi
            formats.append(ext)

format_count = Counter(formats)

print("Jumlah file berdasarkan format:")
print(format_count)

In [ ]:
#cek nilai pixel
dataset_path = "dataset_wajah"

pixel_min = 255
pixel_max = 0
pixel_sum = 0
pixel_count = 0

for folder in os.listdir("../data/extracted"):
    folder_path = os.path.join("../data/extracted", folder)

    if os.path.isdir(folder_path):
        for img_name in os.listdir(folder_path):

            img_path = os.path.join(folder_path, img_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            if img is not None:
                pixel_min = min(pixel_min, img.min())
                pixel_max = max(pixel_max, img.max())

                pixel_sum += img.sum()
                pixel_count += img.size

pixel_mean = pixel_sum / pixel_count

print("Pixel minimum:", pixel_min)
print("Pixel maksimum:", pixel_max)
print("Pixel rata-rata:", pixel_mean)

In [ ]:
#cek gambar rusak
dataset_path = "../data/extracted"

corrupt_images = []

for folder in os.listdir("../data/extracted"):
    folder_path = os.path.join("../data/extracted", folder)

    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):

            file_path = os.path.join(folder_path, file)
            img = cv2.imread(file_path)

            if img is None:
                corrupt_images.append(file_path)

print("Jumlah gambar rusak:", len(corrupt_images))

In [ ]:
#cek gambar non-image
valid_extensions = [".jpg", ".jpeg", ".png"]

non_image_files = []

for folder in os.listdir("../data/extracted"):
    folder_path = os.path.join("../data/extracted", folder)

    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):

            ext = os.path.splitext(file)[1].lower()

            if ext not in valid_extensions:
                non_image_files.append(os.path.join(folder_path, file))

print("Jumlah file non-image:", len(non_image_files))

In [ ]:
#cek jumlah image duplikat
!pip install ImageHash

from PIL import Image
import imagehash
dataset_path = "../data/extracted"

hashes = {}
duplicates = []

for folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder)

    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):

            # hanya proses file gambar
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):

                file_path = os.path.join(folder_path, file)

                try:
                    with Image.open(file_path) as img:
                        img_hash = imagehash.phash(img)

                    if img_hash in hashes:
                        duplicates.append((file_path, hashes[img_hash]))
                    else:
                        hashes[img_hash] = file_path

                except Exception as e:
                    print("Error membaca:", file_path)

print("Jumlah gambar duplikat:", len(duplicates))

for dup in duplicates:
    print("Duplikat :", dup[0])
    print("Asli     :", dup[1])
    print()

In [ ]:
#cek anomali image
dataset_path = "../data/extracted"

anomali_images = []

for folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder)

    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):

            if file.lower().endswith(('.jpg','.jpeg','.png')):

                file_path = os.path.join(folder_path, file)

                img = cv2.imread(file_path)
                if img is None:
                    continue

                img_ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)

                lower = np.array([0,133,77])
                upper = np.array([255,173,127])

                mask = cv2.inRange(img_ycrcb, lower, upper)

                skin_ratio = np.sum(mask) / (img.shape[0] * img.shape[1] * 255)

                # threshold dinaikkan supaya lebih ketat
                if skin_ratio < 0.1:
                    anomali_images.append(file_path)

print("Jumlah gambar kemungkinan bukan kulit:", len(anomali_images))

In [ ]:
for img_path in anomali_images[:10]:
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.imshow(img)
    plt.title(img_path)
    plt.axis("off")
    plt.show()

**Insight:**

**1. Struktur Dataset**

Dataset citra kulit wajah terdiri dari 4 kelas yang merepresentasikan Jenis kulit yang berbeda. Struktur dataset telah tersusun dalam bentuk folder berdasarkan label kelas berjumlah total 882 Gambar sehingga memudahkan proses preprocessing dan pelatihan model klasifikasi.

**2. Ukuran dan Resolusi Gambar**

Dataset memiliki variasi ukuran resolusi gambar yang cukup beragam. Oleh karena itu, diperlukan proses resizing agar seluruh citra memiliki dimensi yang konsisten sebelum digunakan pada model Deep Learning.

**3. Format Data**

Dataset terdiri dari beberapa format file gambar seperti JPG, JPEG, dan PNG. Standarisasi format diperlukan untuk menjaga konsistensi proses preprocessing dan loading data.

**4. Nilai Pixel**

Dataset memiliki variasi nilai pixel yang beragam. Oleh karena itu, diperlukan proses penyamaan nilai pixel agar seluruh citra memiliki nilai pixel yang seragam sebelum digunakan pada model Deep Learning.

**5. Duplikasi Data**

Ditemukan beberapa citra dengan kemiripan identik yang berpotensi menyebabkan data redundancy. Oleh karena itu, dilakukan penghapusan data duplikat untuk menjaga kualitas dataset.

**6. Gambar Rusak, Non-Image, dan Anomali**

Ditemukan 4 citra Anomali yang berpotensi tidak dapat dibaca sehingga perlu dihapus agar tidak mengganggu proses training model. Sedangkan pada dataset tidak ditemukan citra yang rusak atau non-image.

### Data Dictionary

In [ ]:
import pandas as pd

# Data Dictionary
data_dictionary = pd.DataFrame({
    "Nama Variabel": [
        "dataset_path",
        "classes",
        "total_images",
        "sizes",
        "heights",
        "widths",
        "formats",
        "pixel_min",
        "pixel_max",
        "corrupt_images",
        "non_image_files",
        "hashes",
        "safe_duplicates",
        "suspicious",
        "anomali_images",
        "detector",
        "input_folder",
        "output_folder",
        "input_path",
        "output_path",
        "source_dir",
        "output_dir",
        "class_counts",
        "df_counts",
        "image_sizes",
        "brightness_values"
    ],

    "Tipe Data": [
        "String",
        "List",
        "Integer",
        "List",
        "List",
        "List",
        "List",
        "Integer",
        "Integer",
        "List",
        "List",
        "Dictionary",
        "List",
        "List",
        "List",
        "Object",
        "String",
        "String",
        "String",
        "String",
        "String",
        "String",
        "List",
        "DataFrame",
        "List",
        "List"
    ],

    "Deskripsi": [
        "Path lokasi dataset utama gambar jenis kulit",
        "Daftar nama kelas/label jenis kulit",
        "Total seluruh gambar dalam dataset",
        "Menyimpan ukuran gambar",
        "Menyimpan tinggi gambar",
        "Menyimpan lebar gambar",
        "Menyimpan format file gambar",
        "Nilai pixel minimum gambar",
        "Nilai pixel maksimum gambar",
        "Daftar gambar corrupt/rusak",
        "Daftar file non-image",
        "Hash gambar untuk deteksi duplikat",
        "Daftar gambar duplikat aman",
        "Daftar gambar mencurigakan antar kelas",
        "Daftar gambar anomali/outlier",
        "Model MTCNN untuk deteksi wajah",
        "Folder input resize",
        "Folder output resize",
        "Path input preprocessing",
        "Path output preprocessing",
        "Direktori sumber dataset",
        "Direktori hasil split dataset",
        "Jumlah gambar tiap kelas",
        "Dataframe distribusi kelas",
        "Resolusi gambar untuk EDA",
        "Nilai brightness gambar"
    ]
})

print(data_dictionary)

# simpan ke CSV
data_dictionary.to_csv("data_dictionary.csv", index=False)

### Cleaning Data & Feature Engineering

#### Cleaning Data

##### Tujuan

Meningkatkan kualitas dataset sebelum masuk ke tahap preprocessing.

Langkah yang dilakukan:

1. Menghapus Duplicate Image: Menggunakan algoritma ImageHash.

2. Menghapus Outlier

Gambar dengan:

- Brightness terlalu rendah
- Brightness terlalu tinggi
- Resolusi tidak wajar

dihapus karena berpotensi menurunkan kemampuan generalisasi model.

##### Evidence

Hasil pemeriksaan menunjukkan beberapa gambar memiliki:

- Pencahayaan ekstrem
- Kualitas sangat rendah
- Posisi wajah tidak jelas

Sehingga perlu dieliminasi.

In [ ]:
import imagehash
#hapus duplikasi gambar
dataset_path = "../data/extracted"
hashes = {}
safe_duplicates = []   # duplikat dalam kelas sama
suspicious = []        # beda kelas (JANGAN HAPUS)

for folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder)

    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):

            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                file_path = os.path.join(folder_path, file)

                try:
                    # buka gambar TANPA RGB & resize
                    with Image.open(file_path) as img:
                        img_hash = imagehash.phash(img)

                    if img_hash in hashes:
                        existing_path = hashes[img_hash]

                        # ambil label (folder)
                        current_label = folder
                        existing_label = os.path.basename(
                            os.path.dirname(existing_path)
                        )

                        if current_label == existing_label:
                            #aman → duplikat dalam kelas sama
                            safe_duplicates.append((file_path, existing_path))
                        else:
                            #beda kelas → jangan hapus!
                            suspicious.append((file_path, existing_path))

                    else:
                        hashes[img_hash] = file_path

                except Exception as e:
                    print("Error:", file_path)

# OUTPUT
print("\n=== DUPLIKAT AMAN (DALAM KELAS SAMA) ===")
print(len(safe_duplicates))

print("\n=== SUSPICIOUS (BEDA KELAS - CEK MANUAL) ===")
print(len(suspicious))

In [ ]:
from IPython.display import display

def show_image(path):
    try:
        img = Image.open(path)
        display(img)
    except:
        print("Gagal buka:", path)

print("\n=== PREVIEW DUPLIKAT ===")
for dup in safe_duplicates:
    print("Duplikat:")
    show_image(dup[0])
    print("Asli:")
    show_image(dup[1])
    print("="*50)

In [ ]:
# =========================================================
# HAPUS SAFE DUPLICATES
# =========================================================

confirm_all = input(
    f"Yakin hapus {len(safe_duplicates)} safe duplicates? (y/n): "
)

deleted = 0
failed = 0

if confirm_all.lower() == 'y':

    for dup in tqdm(
        safe_duplicates,
        desc="Menghapus duplicate"
    ):

        try:
            os.remove(dup[0])
            deleted += 1

        except:
            failed += 1

    print("\nPenghapusan selesai")

else:
    print("Penghapusan dibatalkan")

print("Total dihapus :", deleted)
print("Gagal dihapus :", failed)

In [ ]:
#hapus outliers gambar
dataset_path = "../data/extracted"

anomali_images = []

for folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder)

    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):

            if file.lower().endswith(('.jpg','.jpeg','.png')):

                file_path = os.path.join(folder_path, file)

                img = cv2.imread(file_path)
                if img is None:
                    continue

                img_ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)

                lower = np.array([0,133,77])
                upper = np.array([255,173,127])

                mask = cv2.inRange(img_ycrcb, lower, upper)

                skin_ratio = np.sum(mask) / (img.shape[0] * img.shape[1] * 255)

                # threshold dinaikkan supaya lebih ketat
                if skin_ratio < 0.1:
                    anomali_images.append(file_path)

print("Jumlah gambar kemungkinan bukan kulit:", len(anomali_images))

In [ ]:
for img_path in anomali_images[:10]:
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.imshow(img)
    plt.title(img_path)
    plt.axis("off")
    plt.show()

In [ ]:
confirm = input("Yakin mau hapus semua gambar anomali? (y/n): ")

if confirm.lower() == 'y':
    for img_path in anomali_images:
        try:
            os.remove(img_path)
            print(f"Dihapus: {img_path}")
        except Exception as e:
            print(f"Gagal hapus {img_path}: {e}")
else:
    print("Penghapusan dibatalkan.")

#### Feature Engineering (cropped face) 

##### Tujuan

Mengisolasi area wajah dan menghilangkan background yang tidak relevan.

Metode yang digunakan:

MTCNN
##### Mengapa MTCNN?

MTCNN mampu mendeteksi:

- Posisi wajah
- Landmark wajah
- Bounding box

dengan tingkat akurasi tinggi pada berbagai kondisi pencahayaan dan pose.

##### Dampak terhadap Model

Tanpa cropping:

Model dapat mempelajari fitur yang tidak relevan seperti:

- Background
- Pakaian
- Objek di sekitar wajah

Dengan cropping:

Model fokus pada karakteristik kulit.

##### Referensi

Bhafiel et al. (2026)

Studi terbaru menunjukkan bahwa MTCNN masih efektif untuk face detection pada berbagai kondisi pencahayaan dan citra dunia nyata, meskipun performanya dapat menurun pada kondisi ekstrem seperti cahaya sangat redup.

In [ ]:
!pip install mtcnn

In [ ]:
#melakukan cropped face
from mtcnn import MTCNN

# init detector
detector = MTCNN()

input_path = "../data/extracted"
output_path = "../data/Dataset_cropped_faces"

os.makedirs(output_path, exist_ok=True)

total_processed = 0
total_failed = 0

for folder in os.listdir(input_path):
    folder_path = os.path.join(input_path, folder)
    save_folder = os.path.join(output_path, folder)
    os.makedirs(save_folder, exist_ok=True)

    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)

        if file.lower().endswith(('.jpg', '.png', '.jpeg')):
            try:
                img = cv2.imread(file_path)
                rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                results = detector.detect_faces(rgb)

                if len(results) > 0:
                    x, y, w, h = results[0]['box']

                    # FIX BUG NEGATIF (sering kejadian)
                    x, y = max(0, x), max(0, y)

                    # Tambahin padding biar gak kepotong
                    padding = 20
                    x1 = max(0, x - padding)
                    y1 = max(0, y - padding)
                    x2 = min(img.shape[1], x + w + padding)
                    y2 = min(img.shape[0], y + h + padding)

                    face = img[y1:y2, x1:x2]

                    save_path = os.path.join(save_folder, file)
                    cv2.imwrite(save_path, face)

                    total_processed += 1

                else:
                    print("Tidak terdeteksi wajah:", file_path)
                    total_failed += 1

            except Exception as e:
                print("Error:", file_path)
                total_failed += 1

print("\n=== SELESAI ===")
print("Berhasil:", total_processed)
print("Gagal   :", total_failed)

#### Feature Engineering (Image Standardization)

##### Tujuan

Menyamakan karakteristik seluruh gambar.

##### Tahapan:

1. Resize

Semua gambar diubah menjadi ukuran seragam.

Contoh:

224 x 224 pixel
Alasan: CNN membutuhkan dimensi input yang konsisten.

2. Color Conversion

Mengubah format warna sesuai kebutuhan preprocessing.

CLAHE Enhancement

Contrast Limited Adaptive Histogram Equalization digunakan untuk:

- Meningkatkan kontras lokal
- Memperjelas tekstur kulit
- Meminimalkan efek pencahayaan tidak merata

##### Referensi

Artikel 2026 oleh cibangsa tentang perbaikan kualitas citra gelap dengan CLAHE menyatakan bahwa metode ini meningkatkan visibilitas objek dan mengontrol noise melalui clip limit, dengan hasil stabil pada dataset citra low-light.

In [ ]:
#resize image

input_folder = "../data/Dataset_cropped_faces"
output_folder = "../data/Dataset_resize"

os.makedirs(output_folder, exist_ok=True)

img_size = 224

for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):

            img_path = os.path.join(root, file)
            img = cv2.imread(img_path)

            if img is None:
                print("Image rusak:", img_path)
                continue

            img_resize = cv2.resize(img, (img_size, img_size))

            relative_path = os.path.relpath(root, input_folder)
            save_folder = os.path.join(output_folder, relative_path)
            os.makedirs(save_folder, exist_ok=True)

            save_path = os.path.join(save_folder, file)
            cv2.imwrite(save_path, img_resize)

In [ ]:
#konversi format warna dan melakukan CLAHE
input_path = "../data/Dataset_resize"
output_path = "../data/Dataset_CLAHE"

os.makedirs(output_path, exist_ok=True)

for folder in os.listdir(input_path):
    folder_path = os.path.join(input_path, folder)

    if os.path.isdir(folder_path):
        output_folder = os.path.join(output_path, folder)
        os.makedirs(output_folder, exist_ok=True)

        for file in os.listdir(folder_path):
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):

                file_path = os.path.join(folder_path, file)
                img = cv2.imread(file_path)

                if img is None:
                    continue

                # 1. Resize
                img = cv2.resize(img, (224, 224))

                # 2. Convert ke YCrCb
                img_ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)

                # 3. Split channel
                Y, Cr, Cb = cv2.split(img_ycrcb)

                # 4. CLAHE (perbaikan pencahayaan lebih stabil)
                clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
                Y_clahe = clahe.apply(Y)

                # 5. Gabungkan kembali
                img_clahe = cv2.merge((Y_clahe, Cr, Cb))

                # 6. Convert balik ke BGR
                final_img = cv2.cvtColor(img_clahe, cv2.COLOR_YCrCb2BGR)

                # 7. Simpan
                save_path = os.path.join(output_folder, file)
                cv2.imwrite(save_path, final_img)

print("Preprocessing CLAHE DONEE")

#### Data Augmentation

###### Tujuan

Mengatasi keterbatasan jumlah data.

Teknik yang digunakan:

- Rotation
- Horizontal Flip
- Zoom
- Shear
- Brightness Adjustment

Mengapa Diperlukan?

Deep Learning memerlukan jumlah data besar agar mampu melakukan generalisasi.

Data augmentation memungkinkan:

- Menambah variasi data
- Mengurangi overfitting
- Meningkatkan robustness model

##### Evidence

Distribusi dataset setelah augmentasi menjadi lebih seimbang dibanding dataset awal.

Referensi

Istighosah et al. (2023)

Data augmentation digunakan untuk menambah variasi data latih, mengurangi overfitting, dan membantu model computer vision melakukan generalisasi lebih baik pada dataset berukuran terbatas.

In [ ]:
#melakukan data augmentation
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img

# =========================
# PATH
# =========================
input_path = "../data/Dataset_CLAHE"
output_path = "../data/Dataset_Augmented"

os.makedirs(output_path, exist_ok=True)

# =========================
# AUGMENTATION
# =========================
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# =========================
# TARGET
# =========================
TARGET_PER_CLASS = 500

total_saved = 0

# =========================
# PROSES PER KELAS
# =========================
for folder in os.listdir(input_path):
    folder_path = os.path.join(input_path, folder)
    save_folder = os.path.join(output_path, folder)

    os.makedirs(save_folder, exist_ok=True)

    files = [f for f in os.listdir(folder_path)
             if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

    current_count = len(files)
    augment_needed = TARGET_PER_CLASS - current_count

    print(f"\n{folder}")
    print(f"Data asli: {current_count}")

    # =========================
    # SIMPAN ORIGINAL
    # =========================
    for file in files:
        file_path = os.path.join(folder_path, file)

        img = load_img(file_path, target_size=(224, 224))
        x = img_to_array(img)

        cv2.imwrite(os.path.join(save_folder, file),
                    cv2.cvtColor(np.uint8(x), cv2.COLOR_RGB2BGR))

    # =========================
    # AUGMENTASI (JIKA KURANG)
    # =========================
    if augment_needed > 0:
        print(f"Butuh augmentasi: {augment_needed}")

        idx = 0
        while augment_needed > 0:
            file = files[idx % len(files)]
            file_path = os.path.join(folder_path, file)

            img = load_img(file_path, target_size=(224, 224))
            x = img_to_array(img)
            x = np.expand_dims(x, axis=0)

            for batch in datagen.flow(x, batch_size=1):
                aug_img = batch[0].astype('uint8')

                save_name = f"aug_{idx}_{augment_needed}.jpg"
                save_path = os.path.join(save_folder, save_name)

                cv2.imwrite(save_path, cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))

                augment_needed -= 1
                total_saved += 1
                break

            idx += 1

    else:
        print("Sudah memenuhi / melebihi 500 (tidak ditambah)")

    # =========================
    # CEK HASIL AKHIR
    # =========================
    final_count = len(os.listdir(save_folder))
    print(f"Total akhir: {final_count}")

print("\n=== SELESAI ===")
print("Total augmentasi dibuat:", total_saved)

In [ ]:
#mengecek distribusi dataset setelah data augmentation
dataset_path = output_path = "../data/Dataset_Augmented"
print("=== JUMLAH DATA PER KELAS ===")

total = 0

for folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder)

    if os.path.isdir(folder_path):
        count = len([
            f for f in os.listdir(folder_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])

        print(f"{folder} : {count} gambar")
        total += count

print("\nTotal seluruh data:", total)

#### Split Data

Mengapa Perlu Dipisahkan?
1. Training Set: Digunakan untuk melatih model.

2. Validation Set: Digunakan untuk tuning hyperparameter.

3. Test Set: Digunakan untuk evaluasi akhir.

Risiko Jika Tidak Dipisahkan:

Evaluasi model menjadi tidak objektif dan rentan terhadap overfitting.

In [ ]:
#melakukan split data
source_dir = "../data/Dataset_Augmented"
output_dir = "../data/Dataset_Split_Data"

# RASIO
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

# buat folder output
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(output_dir, split), exist_ok=True)

# proses per kelas
for class_name in os.listdir(source_dir):
    class_path = os.path.join(source_dir, class_name)

    if not os.path.isdir(class_path):
        continue

    images = [f for f in os.listdir(class_path)
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    random.shuffle(images)

    total = len(images)

    train_end = int(total * train_ratio)
    val_end = int(total * (train_ratio + val_ratio))

    train_imgs = images[:train_end]
    val_imgs = images[train_end:val_end]
    test_imgs = images[val_end:]

    # ======================
    # TRAIN
    # ======================
    for img in train_imgs:
        src = os.path.join(class_path, img)
        dst = os.path.join(output_dir, "train", class_name)
        os.makedirs(dst, exist_ok=True)
        shutil.copy(src, dst)

    # ======================
    # VALIDATION
    # ======================
    for img in val_imgs:
        src = os.path.join(class_path, img)
        dst = os.path.join(output_dir, "val", class_name)
        os.makedirs(dst, exist_ok=True)
        shutil.copy(src, dst)

    # ======================
    # TEST
    # ======================
    for img in test_imgs:
        src = os.path.join(class_path, img)
        dst = os.path.join(output_dir, "test", class_name)
        os.makedirs(dst, exist_ok=True)
        shutil.copy(src, dst)

    print(f"✔️ {class_name} selesai (Total: {total})")

print("\n SPLIT DATA SELESAI!")

In [ ]:
def count_images(path):
    total = 0
    for folder in os.listdir(path):
        folder_path = os.path.join(path, folder)
        if os.path.isdir(folder_path):
            total += len(os.listdir(folder_path))
    return total

print("Train:", count_images(output_dir + "/train"))
print("Val:", count_images(output_dir + "/val"))
print("Test:", count_images(output_dir + "/test"))

## Exploratory Data Analysis (EDA)

##### Tujuan

Memahami karakteristik dataset sebelum model dikembangkan.

Analisis yang dilakukan:

1. Distribusi Kelas: Mengidentifikasi ketidakseimbangan data.

2. Karakteristik Visual

Menganalisis:

- Tekstur kulit
- Warna kulit
- Pola jerawat
- Tingkat kecerahan

3. Brightness Distribution: Menilai konsistensi pencahayaan antar gambar.

4. Resolution Distribution: Menilai homogenitas kualitas gambar.

##### Temuan Utama
- Kelas Acne memiliki jumlah gambar terbesar.
- Kelas tertentu memiliki jumlah sampel lebih sedikit.
- Variasi brightness cukup tinggi.
- Karakteristik visual antar kelas cukup berbeda sehingga berpotensi dipelajari model CNN.

In [ ]:
#load dataset
from PIL import Image

dataset_path = "../data/extracted"

classes = [folder for folder in os.listdir(dataset_path)
           if os.path.isdir(os.path.join(dataset_path, folder))]

print("Daftar Kelas Dataset:")
print(classes)

print("\nJumlah Kelas:", len(classes))

### Explore - Pertanyaan 1

Bagaimana distribusi jenis kulit wajah pada dataset citra kulit yang digunakan dalam pengembangan sistem klasifikasi kulit wajah selama proses analisis data proyek SkinSense AI?

In [ ]:
# =========================================================
# MENGHITUNG JUMLAH GAMBAR TIAP KELAS
# =========================================================

class_counts = []

for kelas in classes:

    class_path = os.path.join(dataset_path, kelas)

    total_images = len(os.listdir(class_path))

    class_counts.append([kelas, total_images])

# Membuat DataFrame
df_counts = pd.DataFrame(class_counts, columns=['Class', 'Total Images'])

# Menampilkan tabel
df_counts

In [ ]:
# =========================================================
# MENAMPILKAN KELAS TERBANYAK & TERSEDIKIT
# =========================================================

max_class = df_counts.loc[df_counts['Total Images'].idxmax()]
min_class = df_counts.loc[df_counts['Total Images'].idxmin()]

print("Kelas dengan jumlah gambar terbanyak:")
print(max_class)

print("\nKelas dengan jumlah gambar tersedikit:")
print(min_class)

**Insight Hasil Analisis Distribusi Data**

1. Identifikasi Dominansi Kelas (Major Class):
Dataset didominasi oleh kelas "berjerawat" dengan total 243 gambar. Ini menunjukkan bahwa dataset memiliki referensi visual yang cukup kuat untuk mengenali tekstur kulit yang memiliki peradangan atau jerawat.

2. Identifikasi Data Terkecil (Minor Class):
Kelas "kering" memiliki jumlah sampel paling sedikit, yaitu 147 gambar. Ada selisih sekitar 96 gambar dibandingkan dengan kelas terbanyak.

3. Kondisi Keseimbangan Data (Data Imbalance):
Meskipun ada perbedaan jumlah, dataset ini bisa dikategorikan "Moderately Imbalanced" (tidak terlalu jomplang). Namun, perbedaan antara 243 dan 147 tetap perlu diperhatikan karena model Deep Learning cenderung lebih pintar mengenali kelas dengan data lebih banyak.

4. Kesiapan Data untuk Modeling:
Total data dari keempat kelas adalah 846 gambar. Jumlah ini tergolong kecil sehingga membutuhkan teknik Data Augmentation (seperti rotasi, zoom, atau flip) untuk memperkaya variasi data agar model tidak overfitting, terutama pada kelas "kering".

### Explore - Pertanyaan 2

Karakteristik visual apa yang paling membedakan setiap jenis dan kondisi kulit wajah pada dataset citra dalam pengembangan model klasifikasi berbasis Deep Learning?

In [ ]:
# =========================================================
# MENAMPILKAN SAMPLE IMAGE TIAP KELAS
# =========================================================

plt.figure(figsize=(15,10))

for i, kelas in enumerate(classes[:6]):

    class_path = os.path.join(dataset_path, kelas)

    image_name = random.choice(os.listdir(class_path))

    image_path = os.path.join(class_path, image_name)

    img = Image.open(image_path)

    plt.subplot(2,3,i+1)

    plt.imshow(img)

    plt.title(kelas)

    plt.axis('off')

plt.tight_layout()

plt.show()

In [ ]:
# =========================================================
# ANALISIS RESOLUSI / UKURAN GAMBAR
# =========================================================

image_sizes = []

for kelas in classes:

    class_path = os.path.join(dataset_path, kelas)

    for image_file in os.listdir(class_path)[:20]:

        image_path = os.path.join(class_path, image_file)

        try:
            img = Image.open(image_path)

            width, height = img.size

            image_sizes.append([kelas, width, height])

        except:
            pass

# Membuat DataFrame
df_sizes = pd.DataFrame(image_sizes, columns=['Class', 'Width', 'Height'])

df_sizes.head()

In [ ]:
# =========================================================
# ANALISIS BRIGHTNESS GAMBAR
# =========================================================

import numpy as np

brightness_data = []

for kelas in classes:

    class_path = os.path.join(dataset_path, kelas)

    for image_file in os.listdir(class_path)[:20]:

        image_path = os.path.join(class_path, image_file)

        try:
            img = Image.open(image_path).convert('L')

            brightness = np.array(img).mean()

            brightness_data.append([kelas, brightness])

        except:
            pass

df_brightness = pd.DataFrame(
    brightness_data,
    columns=['Class', 'Brightness']
)

df_brightness.head()

**Insight Karakteristik Visual Dataset SkinSense AI**
1. Fitur Tekstur dan Morfologi
Setiap kelas memiliki "tanda tangan visual" yang unik yang akan dipelajari oleh model Deep Learning:

- Berjerawat: Memiliki fitur diskrit berupa benjolan (morfologi) dan diskolorisasi kemerahan. Model akan fokus pada deteksi tepi (edge detection) untuk mengidentifikasi batas jerawat.

- Berminyak: Ditandai dengan fitur specular reflection atau pantulan cahaya pada area tertentu (seperti dahi/pipi). Ini adalah fitur intensitas piksel yang tinggi di area terlokalisasi.

- Kering: Teksturnya cenderung kasar dengan pola garis-garis halus atau flaky (mengelupas). Model akan mempelajari fitur tekstur frekuensi tinggi.

- Normal: Memiliki tekstur yang lebih halus (smooth) dengan transisi warna yang gradual tanpa anomali piksel yang tajam.

2. Variabilitas Dimensi Citra (Berdasarkan Tabel df_sizes)
Dilihat dari tabel, ukuran gambar sangat beragam (ada yang 482x460, ada yang 750x1472, hingga 899x1599) sehingga memerlukan resize agar seragam.

3. Konsistensi Pencahayaan (Berdasarkan Tabel df_brightness)
Data kecerahan menunjukkan angka yang bervariasi lebar (dari 91.8 hingga 178.1).
Ada risiko model menjadi bias sehingga perlu melakukan Color/Brightness Normalization agar model fokus pada tekstur kulit, bukan pada terang-gelapnya foto.

## Visualization & Explanatory Analysis

### Explore - Pertanyaan 1

Bagaimana distribusi jenis kulit wajah pada dataset citra kulit yang digunakan dalam pengembangan sistem klasifikasi kulit wajah selama proses analisis data proyek SkinSense AI?

In [ ]:
# =========================================================
# BAR CHART DISTRIBUSI DATASET
# =========================================================

# Mengurutkan dataframe berdasarkan jumlah gambar
df_counts_sorted = df_counts.sort_values(
    by='Total Images',
    ascending=False
).reset_index(drop=True)

# Palette gradasi merah
colors = [
    '#8B0000',  # merah paling gelap
    '#A52A2A',
    '#C94C4C',
    '#E07A7A',
    '#F2A6A6',
    '#F8C8C8',
    '#FFD6D6'
]

# Menyesuaikan jumlah warna dengan jumlah kelas
colors = colors[:len(df_counts_sorted)]

plt.figure(figsize=(10,5))

sns.barplot(
    data=df_counts_sorted,
    x='Class',
    y='Total Images',
    hue='Class',
    palette=colors,
    legend=False
)

plt.title(
    "Distribusi Dataset Jenis dan Kondisi Kulit",
    fontsize=14
)

plt.xlabel("Jenis Kulit")
plt.ylabel("Jumlah Gambar")

plt.xticks(rotation=45)

# Menampilkan angka di atas batang
for index, value in enumerate(df_counts_sorted['Total Images']):
    plt.text(
        index,
        value + 5,
        str(value),
        ha='center'
    )

plt.show()

In [ ]:
# =========================================================
# PIE CHART DISTRIBUSI DATASET
# =========================================================

# Mengurutkan data dari terbesar ke terkecil
df_counts_sorted = df_counts.sort_values(
    by='Total Images',
    ascending=False
)

# Palette gradasi merah
colors = [
    '#8B0000',  # merah paling gelap
    '#A52A2A',
    '#C94C4C',
    '#E07A7A',
    '#F2A6A6',
    '#F8C8C8',
    '#FFD6D6'
]

plt.figure(figsize=(8,8))

plt.pie(
    df_counts_sorted['Total Images'],
    labels=df_counts_sorted['Class'],
    autopct='%1.1f%%',
    colors=colors[:len(df_counts_sorted)],
    startangle=90
)

plt.title(
    "Persentase Distribusi Dataset Kulit",
    fontsize=14
)

plt.show()

### Explore - Pertanyaan 2

Karakteristik visual apa yang paling membedakan setiap jenis dan kondisi kulit wajah pada dataset citra dalam pengembangan model klasifikasi berbasis Deep Learning?

In [ ]:
# =========================================================
# VISUALISASI RESOLUSI GAMBAR
# =========================================================

# Palette turunan merah
red_palette = [
    '#8B0000',
    '#A52A2A',
    '#C94C4C',
    '#E07A7A',
    '#F2A6A6',
    '#F8C8C8',
    '#FFD6D6'
]

plt.figure(figsize=(8,5))

sns.scatterplot(
    data=df_sizes,
    x='Width',
    y='Height',
    hue='Class',
    palette=red_palette[:df_sizes['Class'].nunique()],
    s=80
)

plt.title(
    "Distribusi Resolusi Gambar Dataset",
    fontsize=14
)

plt.xlabel("Width")
plt.ylabel("Height")

plt.legend(
    title='Class',
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)

plt.show()

In [ ]:
# =========================================================
# VISUALISASI BRIGHTNESS
# =========================================================

# Palette gradasi merah
red_palette = [
    '#8B0000',
    '#A52A2A',
    '#C94C4C',
    '#E07A7A',
    '#F2A6A6',
    '#F8C8C8',
    '#FFD6D6'
]

plt.figure(figsize=(10,5))

sns.boxplot(
    data=df_brightness,
    x='Class',
    y='Brightness',
    palette=red_palette[:df_brightness['Class'].nunique()]
)

plt.title(
    "Distribusi Brightness Tiap Jenis Kulit",
    fontsize=14
)

plt.xlabel("Jenis Kulit")
plt.ylabel("Brightness")

plt.xticks(rotation=45)

plt.show()

## Conclusion

**- Conclution pertanyaan 1**

Bagaimana distribusi jenis kulit wajah pada dataset citra kulit yang digunakan dalam pengembangan sistem klasifikasi kulit wajah selama proses analisis data proyek SkinSense AI?

Distribusi dataset SkinSense AI menunjukkan bahwa kelas Berjerawat merupakan kategori dengan jumlah sampel tertinggi (243 citra), diikuti oleh Normal (231 citra) dan Berminyak (225 citra). Sementara itu, kelas Kering memiliki jumlah sampel terendah yaitu 147 citra. Berdasarkan temuan ini, proses pengembangan model selanjutnya akan menerapkan strategi augmentation guna memastikan model tetap objektif dan akurat dalam mendeteksi jenis kulit yang jumlah datanya lebih sedikit (minoritas).

**- Conclution pertanyaan 2**

Karakteristik visual apa yang paling membedakan setiap jenis dan kondisi kulit wajah pada dataset citra dalam pengembangan model klasifikasi berbasis Deep Learning?

Karakteristik visual utama yang membedakan antar kelas terletak pada variasi tekstur dan intensitas cahaya piksel. Kelas Berjerawat dan Kering dibedakan melalui fitur tekstur mikroskopis (benjolan dan kekasaran), sedangkan kelas Berminyak diidentifikasi melalui pantulan cahaya (specular highlight). Namun, ditemukan variabilitas yang tinggi pada dimensi citra dan tingkat kecerahan (rentang 91-178), sehingga tahap normalisasi kecerahan dan standardisasi input size menjadi faktor kunci agar model Deep Learning dapat mengekstraksi fitur morfologi kulit secara konsisten tanpa terganggu oleh perbedaan kualitas teknis foto.

## Final Conclusion

Berdasarkan seluruh tahapan analisis, dataset berhasil dipersiapkan untuk pengembangan sistem klasifikasi jenis kulit berbasis Deep Learning.

Proses cleaning, face cropping, image enhancement, dan augmentation terbukti meningkatkan kualitas data sehingga model memiliki peluang lebih besar untuk mencapai performa yang optimal.

Karakteristik visual antar kelas menunjukkan perbedaan yang cukup signifikan sehingga klasifikasi otomatis menggunakan CNN layak untuk dikembangkan.

## Recommended Action Items

#### Jangka Pendek
1. Melatih model CNN baseline menggunakan MobileNetV2.
2. Mengevaluasi performa menggunakan Accuracy, Precision, Recall, dan F1-Score.
3. Membuat confusion matrix untuk mengidentifikasi kelas yang sulit dibedakan.
#### Jangka Menengah
1. Menambah jumlah data untuk kelas minoritas.
2. Mengumpulkan data dengan variasi pencahayaan yang lebih beragam.
3. Mengimplementasikan transfer learning menggunakan: MobileNetV2, EfficientNetB0, ResNet50
#### Jangka Panjang
1. Mengembangkan sistem rekomendasi skincare personalisasi.
2. Mengintegrasikan model ke aplikasi mobile.
3. Mengimplementasikan real-time facial skin analysis menggunakan kamera smartphone.

## Referensi

Mohammed, S., Budach, L., Feuerpfeil, M., Ihde, N., Nathansen, A., Noack, N., Patzlaff, H., Naumann, F., & Harmouch, H. (2022). The Effects of Data Quality on Machine Learning Performance on Tabular Data. arXiv. https://arxiv.org/abs/2207.14529

AI TecServ. (2024, March 5). The impact of data leakage on Machine Learning Models. https://www.aitecserv.pt/en/media/news/impact-data-leakage-machine-learning-models-032024

Prosemnas LPPM Universitas Aisyiyah Yogyakarta. (2026). Analisis perbandingan akurasi algoritma Haar Cascade dan MTCNN [Proceeding]. https://proceeding.unisayogya.ac.id/index.php/prosemnaslppm/article/view/2120

Kohesi. (2026, May 22). Perbaikan kualitas citra digital pada gambar yang gelap untuk mengidentifikasi target dengan metode CLAHE (Contrast Limited Adaptive Histogram Equalization). https://cibangsa.com/index.php/kohesi/article/view/10546?articlesBySimilarityPage=2

Istighosah, M., Sunyoto, A., & Hidayat, T. (2023). Image Augmentation for BreaKHis Medical Data using Convolutional Neural Networks. Sinkron: Jurnal dan Penelitian Teknik Informatika. https://jurnal.polgan.ac.id/index.php/sinkron/article/view/12878

